# Voicebox GPU worker

Turns this Colab runtime into the voice engine behind your **voicebox MCP** (Kokoro, Chatterbox, Orpheus, Dia).

**Once:** open the 🔑 *Secrets* panel (left sidebar), add and enable notebook access for:

| Secret | Value |
|---|---|
| `WORKER_SECRET` | the `WORKER_SECRET` line from `.secrets.local` |
| `REGISTER_URL` | `https://<your-vercel-app>/api/register` |
| `PREWARM` *(optional)* | engines to load at start, e.g. `orpheus,chatterbox` |

**Every session:** Runtime → Change runtime type → **L4** (or A100) → *Run all*. The last cell stays busy while the worker serves;
stop it (or disconnect the runtime) when you're done to save compute units.

In [ ]:
# 1 · GPU check + unpack the worker (embedded, version 1.0.0)
import base64, io, os, subprocess, tarfile
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "No GPU! Runtime → Change runtime type → L4")
APP = "/content/voicebox/app"
os.makedirs(APP, exist_ok=True)
BUNDLE = "H4sIAO2My2oC/+19aXfbRpZof+avqEZPjwGHhEhtTpjQfbwosV9syyPZmXmj5iAgAZKwQAABQC3W8He97++XvbtUAYWFEpW2c/qdEc+xhaX2uvu9dTGNo1kwt5PrP329Xx9+h/v79Bd+tb+Dvf7urnomnz85ONz9k+j/6Q/4rbLcTaHLP/3P/BmGcRTNg8gXUzd3w3i+8odiEV+KPBZBBGsThsJ3pwvhU6muuFy4uQhyMY2zPIMi4qf3H8XSX8bpdVe4kQfvMpG4qbv0cz/N7E7n6MJPr/NFEM1FvvDFs/evhX+VxJmfCXcSr3JxEQdTvIHKZUXoYAkPZ2m8FAs/hZ6zGOoH8CgIfQF/sbEMWg39Thav0qkv4pnI01W+ELM4FZ4/c1dhzu2mbjSH1kwvnmY7MAJ76WETcz/yUzf3Pe6HWoT+xYUbrvzMsjuwPJ1gmcRpLuKs0/nl6OT09fE7MRLGwO7bfaPz6vjtEdzGme1HF0EaR/bcz03jl+PXL46eH/+Hg++NrjB2pnGU+1G+Q5OdxFeGBa29++X0tspUAGpDgcTNF/anOIhMbBEavIAqmWFZnc5fxGkSn/uRmFyL4zRZ+KsMd2+a+jAzmpMx85du6Bs7Bv0RqT+DFY1gxaZhkOBKwp6+DFxaqhdwAxsAQ4S3ceTbnZOjH50PR//xAYZqGq+g5iO5+Dktq+C95b1cuklmiw8L/1rEUXgNWxcmIpiJ63glziOAK//Knebw4hI3FR8/gj9GR2g/A8Au8qBNBigaXb5Ko0xgDzC1iS9cka3SJA0IAKAbNwXgSeNJ6C9tWNr3J0enRx8cGDku8I1agCFMIOYlwk0BzHcNCy7aXnpuZFhrAN93P71+d0TN0CiN8/g8TmMoflOM2kAIgSffDvqDbvl0nqyc+QSeA6h0hfgLLEl0la6iPAAYiyPx4v3HIWNZlAsvdWd51hWRD/iC0J8AHmQEyr+cPHur9eVOz12AZmhXjqWH7QJ+rCKPkCNaLZNrQxsI7Q2W/5nLf7v71hY/ulnehT323agrZiEssueHASKrLZ77Wc5YRKPifQiWsNi23i5jLjR8ZrgzZ+G7sAqwdHA98cPQlddRMI1hffnGjX1PXcNY1GUGW7FQRcIwvlZ14wtX61GHEqx1rsp98rMsmKoeU5wGXS+dmR+lgbpZBtOF64fyLllNz+WlP13E6jINppu6XDph4C5lwTi6vpKXGeygK69dj0tMZo6/XLryMsjcYk0mOEdYOXkdBuH1hg4nMH53wosH13M/TufqJvQvg0xeA7AGPK0Ftp0sXHk98XlcCxju0p+r6yQLNvQ4g1UNZMMB7wtdLnkb6eaT1oc/c7yYy/gw99CnJUnKp4l8OtbgJkn9zM8RcDTUrIBQ8azcs7XeAPKJrIKDTDkS3/eo2fw6oQYAFyYMC5IhwMMBoqMB0EyoeUDdXcH1Lj0vkAUac8+RtCGL0LunrkKgfZWegBxC4WpPhqE3mCyAni6Dz4DfMQBoGnjA1nx7bgN37a0A9+HPfNIVi0DvTV7KP8a0oM+bSNBuGwnah7m10pBHZYO9HFDdgJ1ZJXkch9kP3w6MR21kROMRsH7PhXkC+7kEOBXPXnfF29cfLFscLeM8ACLnBW4ovhEw6biXLeKcuAos1Pdid0/gIq5wIEAGp6EtXgVA+DeRGLWs3YKhKUCBvz+40bXG2CJg40+3gLlaQ0P5dxtYA042n6P4ALO8E+QkmEmQ61dArg0PdaKjVt2XKxqgJJEF+TU2tXsAklu47GIPMGvgk26IN99+g7R7CaObNkB3Ops7l34wX+S/e9yDO8ZdjjqMLwHeRyKji64ASdEnTjMh0QtEvimiTX2MuQ8MMMX5+FsM8lt9kBWEbkNbhLi7UdePKsj7+vQYuLLnfy+iOOqB0BwG2UKsUIxFYWgJtYIwwMZDmKNXJVhMnOq0CTdyXp/Mk82or6STDXi/14b3323CeyOPU5DrUYbLgNMvUeZ2p1M/5G3JIndaihStwoQSNvee2+ItqANisVq6UY8qEYIHUYiaRe7OM/FD6K7mi6fih+kCuG7ow1UW8IOYX2RRMJvRi3kauxH8vXYv8c/czZKnm4hCLhkUiDEkP6AgwPexkuIINqjQZ3dKf2J/C9KgWlb32NQ2dOF+kHuoQe5AQ6+DBkLEiZNs0eB3BxtabKBC6oOIGSBJcRI/csP8egvOOShaH9yLjNWB5+kIGwNpF8RBj0RNEIORg8D7BIrUBwv9OJF/6eSo6mR3o9Hg237J5Hf75VD3+/3+F8dMYHKbsHK/DSufbMTKeZB/s8jzJBvu7MD1YjWxQRfYidw06IXuJNuBrmx4cQduojY3sA+BNb+DmuIN1ASeDE9Jxe8BowARWZhnp4Pxztnp7hi0ICRrIJVM3DATYXAObwllM0uYiKmZ9T0rhFIpd0tuuwk3t+LS4hJmCa0RIZqmQZJ/Zb6N3C+bcq3boX1PkxN1aD9o4tJ9kH5g720SP78I0h/chvQ4e4AakN4cbPz8brDfL9seaALAl8UiULTfH59+cF4e/fjs45sPmroNrHkJ1T6kKxCWtd9fBL4RQPeR2ewA+BD7FRmgBMKWee77SSYO+2KZoWXIDVKrbNHxEA17+wcKEY0IeCBoZZ99eDVDsOntSa0d1YCefJ0h/1+FIZohsAPi/WwbewRoA7p7yM1l7jIJfYdUB1hDoEb9Lg8b4JneEdG7AB0gRtXaR1aZycZ56IykuDQdWD6sF4cXvsNwbSqD3BzKRNaQezWMtz5oiCiWpGxMy0jVqFrDpiEMAGeAg/d8uE19aSOz0eKFTaGpBfZgTTc40vOuAIVoiiY/aRI54yGMzxSqje0A8CAz5Wjwd4FGIxqigDZu1haZuGRbpUg/tooawUy+IygaY3+mBvUKpiyayQVa8SJQKt4BYRpWwBF7noWxm5sXVqPR0ajAJOGHmY8yNZRrNACwbmJFNswhEnTFBdBKuNKfu1fy+YVVtgEreHYOXYkLepT6aMPCp3I/k+sc9EEH1lbupVw2pRo2LYOz0jT4/n87N1zLXiVAeUxrbViVflQrsOwV+yEZFbuFPdeY0KQMHg208aeH3/+gH4OBswDx/as5gW73//T393YHVf/PoH8Ijx78P3+M/+dkFWXi+N2RJAlokEa6gX6c+DICDpXmoNMCHSKKC4zlAs01ORqwwxjkKIKdDtAeJiGiClKi15PtSu0VHpA/BRVWZvpiJ7uOQAYEwcG/IvMOyZBElZixdFFZmTlIxIy/yRsqizdAUfnNWvSeQiPeStpkulUuvO78dPRBiJ0UJIZrLkqXwJ5zEC7WnWN0WAA59Mh/Mth9gi4eezAkNjmHJi7da+V4IueGcmqhf8INz6kaTzaruI7cdA4TyYDa8oMwmHTFpyyO0LUDrPAa/ssXKUsyXZIGuigPT/0JaAYd9oGBTmDT4qeyFfHczfxXHz68P/F/W/lZ/gq2J0QLywfVFL48pSqdDnTCTAC2109zs1+6lbwgRUHcVPegK+Bf00FB0XccC71MbgL8SE3EfpbOV0s/yt/jHTAfeG27nue48rlpqG2HXUhhfAFIGCOU4VqLkrYEUwbmPAJO3KySzjPsPrGpe6wLUkans4w9eFwsqs1XDjxehT6wS7UZN1jB5jvkk3kfquEy2/gftCRBdISWG5sdovA0SVEsmBlnlfpjAHsXlVaQTG60RkRP5P2hPZitEWBn4SpbyOEDmpxjh2pf7DfwACqQR8gXBP1+BpAFgOTSuEDmTBU+mj+9/whz7YCYlmXildm+7VJ4QMECVDxn6WcZqJRm5oezrnjsaiJZAs10isJO5keeLIbmLQCLySet9CQGZBkRtNreaplkJr6HtcDCZintYAs2tuWAnJrEAGQmlmgrsIBVAKAxXrBLtPcBZTI0tSVJGEwJd3ewP2Obym9glXK0/GR5agKVMnHAllWrWtbM6mO+RCC3L1MQXbluuTZe7ADJoMXRVoRq8bKRXaEkI7jbaxY2sQwiEUmaTHKkpKlV3+/vY3UQ1OKUdLsYnW6g1hvr6iiQStaHUenlz9gLwZFRlYKlNLhtp2W139SmI7Bn1Lmd0lLhZExEDHomV5UF4cam9GEnUAKdGDdrbTvz9Lo6TLIBIJoMG1Ykd+UFMWwv2pClxEsTNWGMZ8wxxkQweAjMOiztieQhFisgW5mpikXgFgreozdb8CCrqjZI4lx6Y91MZLNKkWwm4Y2mUPAvmEYx2WqbTZArmNxQINBTPUvsQM1uXffMUm1j/aupn+TiiP6gMwFG5w9ZLcVxdwHcoke5mKZuttCU28pwCtZkE4V0oFFz44APeMAK3mbGDZJ507dsx0G24zjrobgBsoww32nhXaZpFLwYQIpIMQ4VNuOVxSwRlSh0m5sP6ss9f1Ks+aoBYLfL//tPnuw14r+ePDl4kP//IPlfRSQpEZfl3cs4PffTR5lIVhPgygLxEWO3QNI/hRUD5UAKV6gHeP4SdQOTIsVALCp0B9AagJr5fk0pANQF2QWkF/EpnmQc2eVTJJkXU4t+RwYVKYGoKxKo2kvSeAqSDUaLIcVrM6F1dTUFZBGyeoHohV4mkQXzCCS3jydvMl1lKbGg0E4AEEtyZv778cnPRyfO6dGLE9AhlgA0GISU+TnQrGerfDEUSH6uYZpeEgNJJL9GJn7Fd3EafCZaPRTPfTcFwe5fKs392lVUeQf5RbbzWFwuAlhIbsPtXLhh4Im/+VfJ6F9hAiOaBZmahQlL58FKI5MWMH3QQaZuhGNbwCrATNHIh6581lLgHdB2GLaQbNvaoKdMQMY8BDlhAVyAtJXF0p3qOgsF5C1WeRDC39VE7sudukxXrFaBp/pbpSFK7cQauuouZbG2Kz4HCa4Hqz/TOAz9Ka5ipjjscQoz8L2XwTT/IioStSEHQcugGmGd47eMhkg3/7A6JVueUvQt8uAXyIKj+Dd3KI72+7uAZAxqzejACuyg2AxylYxCfGHjRed/HT8/dV6+PukKjH/jqzfHP9EVmmNbwgk9i6O92NqKSCl1bvalxnMKNSyL3NaFMmNm9tI992E1MtNDEA+y3InPpUr06uhExk1uuWad0w/PTj4cvawpbq9fOtRQ6qObLIHCZmr819mz3n+6vc/93nfO33vjm0H3cH/9L4YlDa8wHbPQiFjHoyZBg5hRs8ZfXw3/+nb411MU+B67VWVOGm+JKjoAEMraXZFplRE2AdmsLXqzrA7ryhK1hf0HiRSkJEU4Pj1C5Gg2XAOK9x+fv3n9wgG6JiGCR4mEwsTOAQXzcHQoHou9w35fjhgICmrPavq8opb4BstSAagOBRD17ci/NBnkCtWvi7IcNr4e3kBTa0N7I+mGnS3c3YNDC7SEKy+YAwaa1tlwvz/WLdXYiraaa26TyB21SzTvBv5bG3Ja04U/PXfgiZzbb1nbHpTT+y07M+DOGJ/1x9b3clr4EK7oob7k5s/+9RFTpF8wDFhevwZyekXXVmM3fnRBsaOH85gsEl97yXBqT3U8IJ5HvSIWAJNxZF2YYZdGhSDxF+BuX+WHrJt2Bh3cQTSLW5GCXUolt7B5H9nhb54ZANBe4PayZYBA3OsByU6ve9DkCGlDl4PM7TzO3VDdrDLfM7ZW6aBNjHRx89E0u+hGMTNBuFhFILGg/oUK3YhdjbiwMLTRoF9gpp0lYZCTlAGbIrcEfzQ+wSND3xvCwNmVqkak9QrpJrTHbZhG17DGdTi6MbAhUJK4PYMa5ICBFNVJk/1Z9BiVvUF/F3j0ACOYsdNmSXyqF1x32nTA4cZxGBHap8wX7z9aRnU8HAtW9gr3668KYVLalFYwNtFlhdezIpKKUkoM3c9BeP09SGcgJbDlNvRdECRTf+pHeXjdo93CWV4u/IgirUH0XYUeOdBgFS/JI1qaywC8g9xxWo1C2C/aKTXJBK18yNkx0KH3VLxHjtBFUHhz8lHEWK5m48ox+GokbiL0n3sYrY/Qg0F/wN6l43VdM24hVcpKb23xomF3PGHDY7XUZOUBDym8pZtPI8DaOD89Z0NKiehnJViM8U0fDaED+0BbM3mQRFoZcS10c+Q1SS6aN5QK6HYuxUFJhshM0B9ajVzJdctanmFr6IU15Cgw0K90T6NbY7MYou6hx3JAID7MsE6DtysBiKg81lv3ZJ+wE3M6aODqNs2SEKarCOjf6sKQBzuY/kmXLFzv2QM0fbA6ZfR+Q1JFxFPSqiz3kFThyOgGIIJuNnY2e7S6EEmQFGd8VG/iJrlei95v4qaANl7BszJKabx+hMK/H4aye20om0jx3SMsN7EEHEJbgolW0EEFiYCnNlIZZjXWYYjK/jASfYRRHe7xWRs4FY9AGQtB11stTa0jrZcSQUsigFIU9fhU76sWoABzC5Yw/giYjmlGbe3gyDf0+hQwrUtxDzX744wCIrj1pjm1NjkJ0OgroRqgc99w1TXqjijBA0cBxQgHx0BttJj6sjxOTK6nWa79KAMltX3vyOJbUKnqOFFuK9eAyFCVJsh5JnxmzE5AOQRCG2QtYSBV4mwvgaw7eeygZbLZ5oYFapirC7rFdk5F39ob5DIlKN9SqkaxUKuvkKvfQYJaSU8hIhP1u3n8uCT6ULJ6eG0o8M+6Of+qLEeczTxLruvH1UDRw7MhFfsPDUjzEJK809laiJPuQvT4NKkUvhtb481EEWNfLkZESqeX3ggHaN0CMsV2JI1CHrDVkP2Guiz+jdjrg6Z12G9UYFKil/2haGTYOn+Ecg2+20OdKuDrBpkvTviQGekrpgIF1MHxrCOPQgKXMNE8hxr+TgkwVgusbEQE9avab2y4JZCYGWiXGe7slA71Bm95hLv2aLyWjrJS+N61NnbXhjBc/fYqLC3BmiYmi9lNEtokjnIBuf3NhVtoxybHy+Z1ZFtE6PuJ2TL92/bXCzyCkImPh2d5tMhPABwxdK3aWLsnaCu6xL6czi3LW5RFHN1M7Yo9wIBHh3qRVpbWaZechaqWfKXSAGnjv4/T3AYU9+A3iZ37Kaw4jMZsTmUjEiX2pQuq4W6/hVDzZmn09gNjyNFVgnESmxo8D3CkbfJBuWDb7TnpItoGkAtW34HEvUaSUteJpAxQZXrsXq4RDGmjvR/BYJ/3lrzDc3N3pEUyqCHrVhi2i49uqtEJw5bohPVd/uwNBBHuNevCAXEKC7Ev3SSt6Y74VFoLGy5d3Zhuo1mbiEMLVrdQEK196d/XvPasBjLOdyVCY0QQnoyGXZSquPl1DUzFmYFMxkKzRz4zNbGytI2W4kdpni4Eokv3AuWPu4rJCBS2OMISONCn3htGznGHJrToSCtrcYiBrT7aIQlOc9AVk1UQeioZAh9j2MH/hDzoUPrTilhsaL8rln7ukqG7MvOOhtI1NRlqsV2QjA9kP9COxWt2UhkJTOMjdUorxvjWqXLDYgo8ZG2Oj3jxHjECyAjl9fCG2tY1Bxmbjr1tiGu/UecJ9vS4FPcS6sDMQMgybLij3SxeA0TK4JCyFYrRGEI/Ko1BEV84VFtSHhXhCz2ocIh9aiNAN6SkwSa86fLm3FSPKVSPGHRbDxc04jSQl2gdwW6mPqor2E35uCBfJvapoK0+QU6IweEW9U1Yd1lxQICCopcqgGUzFGkGQqIdpV0cgGEVYV6HqA7oJSiYrodZQRgkON4THUtKxiwnoUccmcUQLUmAtMlaLUYhKsoRVriQOkGgHSqrS5TGGESF0jLwxplc5+hALsuOSA4Y6utDHid76ebThVnTaRurY9SOOinH8aB3uC+mCzelkymhj4epoV8vmAeYAqKHxNeR63MX8ufLREMJEPPY+4bLB69wgydAwzneSJslV9ahmUp/DWguIRm64Ecbobi80aHYWCUIFJtBd3PM1QbTuhY/xQbzbGaTPRPh3VZvu2IXNdWFmzmVUU7iODQ1Wv91Td+0a+x11LYrS6cAMegox6cKRnkhKAEILkKU3LU+FdFFj7WDBSFJgPuh0FyDLMR7uxiFFV6615mz60lx+w7HwjQmSwO0RVBaIdoNO+hstkx8Oo3dI0uoEj2MHmX76KF7iEaFrVYsodatM8Hi206lCrdYsyCS1LQdecFSPBWD2uJhlDK9X/puZLpAm0aDoiZulCJmgF8a5cgXONBBXzx+LKTvploY0a+L6GehP0dXEQLvCm2IoDxgRh0TLtxJpgICn2LLFVdVMKOAQahVM6InrifdpSBzo6e4HoRYmd8Znozqd7F3aB0oVoKiP52LKuMR+XVvMIZNx/c8itxN2d9QzlAnHcVqyXKUUwp91HRrYahrpknKDdGbwjiWGBQroX4+9ZoFsmmQXNsUSBOqgupgngP63XWlCvqLoRlTG0d34/pUmjElIOoz2NkRcwJNvLBsN6OAyAIiG5L9axpeLRCA7N9yw5iCaXGgj6v94VDbxwrQggfq0gThJgyiLHGnPu6r1lhPDEDHwwPCAFp0SFDrypKs87ZpEBJqA2qiQ42hbDB14TnMwk0Fg0EQrIC7ZSnwlkMncQDAWYd+auVp3eZfAW5YP7OBi2f1UY4lJsL/2CZPqwjtJcpcUCEKkmLC8/7FW2dwWD2sVwvh/arcBIN7KESo7qjs/NvHo48YS3M27vzbi+N3LyuuQ1CHPcoVYFq186gk7pkVLUKSFjovqvtn5IFReqnUDf4LAo86060vDB/aVEfAxwQtUmjnjuRY0E/hwMw4gl+Gt6gzHfhMKrFscC4gUBZBUNP9q7fKu7KOEuXQdQwy3E0IUmhpjS5OY1KwQGUM9ISdqGdjXbDkovAYoYFukH7v9vu3CZhciQaDGeZcmMagB3UEncSVg6AcX7yx6jRvAChOviYfZAXK+CG7LDvDoEM24IXSFRx4PG5QKgIg6YNhf9fTlboWCRkaoSr4XG9KnfTC8CcZJtFmpaisPJnab4L1UEVGwgQ8YUrxeYel553ejiPV3jjq+cskvxYcpl8KI7gctpsk6AS6wVkNcbIo6LKuyANVhwuKAZ4NYWH7YxQ8g8iLL7GanI58YG0yRJUapqpRPZUgJWWAYEpXR0PCSEkb/9s3KRrobDjYHReuFChQHOFV6mw7RpagJ89GVEdZDq0sWDsygeksMsoV9PjxC7tyTr8LVNLUKmI5Wc2qnrWQgD9UGecwy19OyWwMigX2EBw4aSHOXnOSUCQixfFiVolxIRGyklHtZOqCshXCK5LlMHYR067JPdWQkJ7SQAEEz4bf9sfrRtRixUJURjzCJp3hDmHOjFpUY2H6I/pZAjRWPivqoT0Vboq3RHcVPJatl++xMRswKJhdS2OudOH4PK4GncCHaMxGImCGwJl10VbZFGEEkpCvIqKd8K9COkuTEA1Kwt1YrgADkwpLY71UqqiIN3BZ6OElpaXwUi2DWZdTp8jD/NQbHmUqskVoFuOZNLFpHSirnGQFBVfB84EFZ6l1LrM3dstsSnWrwSfYfC0mZOPuF7SUCSm9YBgfV05p0QsJl+MqkaNXEg/Iys7FQsCA2yIStjCePX5MbavEDIimSNHUfcOkVpktrgBZQqHKo8B7NF4XpjbN5l0/Q7mFEa4kqnUjHG5ZiyHu1hRjxbGsoQYe5bmsYQEqt9nzQNWXqkjFqnfXekgjMi8yEr3xnZY74BHQU51jdCqwoOjcuM6eIga6kkdVlrJhzYAHXbFn3ekXKTkZXGBevYCILOICgi4OWXq5GElwtX5QIji83aKHzAfBxMuKodVPzFJcolWzMCMjvMEV4eUuLMk70ogpWBYo9kPcwMiG9i6evdXIsDxCYP9nkPyIoeEbcBp2tuzLhkoUn3BpkFfmc4myiOxpgevFXlVx+rNUANohKK0BUONRp9YO8i1j6UbBDH1I5JTo6qdxYSjOReBf4gUdUMgkrUed0/Mj9KEXwkWV2HiwrSqkmo/7OGEcF5IY8xhkbDVfao25lWVRyCNm1nRkMQ8j/2ZVE/1EUiazQHS/9isYAa+IlyHKQMmGqIlFYFNuIbEoFQfRyu+00txucZui2MFBBKso4myAbdStYW/QuWfnHs71uw9Rto2UJR/NC9+tu9ip2CyIgmwhZ6RPg3e7BWhIgOERUkzQ+ZBaOufwtnNmngEJaUXYTpmnQY5SN9ZJEYvFqouS9bSxrTb5QpW3igaIzCpphO+Kd2k8TzHz35jkZ4JsPhlbRVVLRTHrbyXTlkNVHLvElFEhoWp5jc74mQPjILWYemYwRry70gQ5VJWKZvEJLiYr22ygkDJUVlXRbqU0yLQ8VNvTCk7gVtbA88zAUyYUi4GnQWaGPGWmkbydTTQIOlaMKC32oRgN27o6WtckxjVXj7dDXzugsc6dw2JKXCagktKjVBEY6lJ5WQC8VZe9OOFUTbu84MxMzJEYzpEP6fLxxVe1wqCDns+yqLT7iuw2Em+xx0Ll3ipMDM1cW5hwSsVrIP6S8aSKwJwfsFtk6euW+fUq2VZklKm1bjbOe0tJtFjUK+NElC6AgJzFuDXm7GzY2+chzOgERGajpQTYYeFzJ6MdpWjIkL2Y7B6wrKr7Rsb/A97CSFjMkwEBLOhlxVScyitpl9Fc2jWZhSiJo/KjkSO1ouGq9GsL3w1hdNVYgxsjPi/9ZBd+mrEU9sKWnwrAQxIJ0l4Hm66dtQL5R55qs3jNoUgZWF8b5gXsjcNxxPpBD9wCfipPgpQLVexOGX/J0VnyHd/U8+NSbhO1cFiMA5q7kghKwknkqz5ExTlBS/8kKR1u/CfceGLj/F0Fk3b8U41IqMrj6vG4YeWo3brMg8JHOW/PhsKHQR25L8g01WHnHQOocrFLnd+ZOwXlsIfcKa25Uxx3lQM9rp+Tmce5CrmrZA6pHJSWZxhrQVDkdOUDdZvPu0H7KFTLg9Y3XBxtlOXAcKD1YSlPynYpTSpSqHiKOSYf0xkr+ecOS6YhI8LwZLYI0SfSnOwtqVciPaPKHblqVhxmR4eWzSJdTFVkYDWFeKkkwJIVG9YtuWSwiLlqCOT8Uu79bfUJdfb7g0oumlXkSjjwK+loimFyOh0mxcZdrVOqFEW2Nzam8tVu05jGrDe2R4ent2rthg9aI7Us5PGKCldST8puksGSVsioNT4b7vV1Y86Sj0Kz7T01/otGs2Oeub1Zv/fd+Gawu7b+BYB41YSDJeHU0p4DX0nMgaWI9lZTKSZAls6ykbHVgOzK9v8D+Y+2AO2vB5ANLVCSem6KKEwj6vYOINm0urva6lbdXDWf7jZ7v8NKchsI3AsMCqO2ttWlBg4DQYmoefxhg/VA777S8QZjQtVwLm1tZc3N8fnNIW9hfP0Hwb5l9ymvQnNaUKr0UMhQrcP9OhXWg5nRsUiZOmwo6PkkVmATZ1r1cUs0eVjpirIQtHdyWxwzd0RaXBnPvDmUuTZuGcd8UGOdbYPNNhx3aTJWlHJEMXVkkzS3rbYTEF+FBJZLQ0FjJI10aeC6N00PSLwLFaVQTtj4efyNtWMib//vFSVuse6Jji1+6hYM2rUI2LgLo30FWUOIk1uRZ/MGlNEBnE9FBniMSLmguH9O1zXS2u/SSgLKRmxyYsHDtLZFuZs2BWdt3UnXt0h6J412JUC1Bc+3NF3Jb8bWuPUXNgU2u71XWrVC9iWpjRWWVdVSgitS5tVgaOwWqW/gAWVjsLbgoHuVlZ64HuKhz+dUytRFxm1SC8ufVdZl/m0IT1oSu1h/R4PBf/8dzUWbhBuc3XL4JYBEujk3eBI0OMctKXFRWtZqCErmv81OibI13Ra2OVabpv1FZtmikQL23UshRSq8g5OmoCsaZc3Cw9OvqK7VWW6puapVANKTgaDGy2Bt08zLICvst7Dij9w8d6cLzEr7PX1mEhFpZNyoDpDXcj4A7GFtPNpKUyYeShxTeSgn7FmaVbeK02mBQptcY9/x5JOJB1cLPbuMdQI4kdt8R1ZeLnJ3ul1p/Rph1rONtfDsrlGW1ZIyyiotGXq1iFOluf8gdAUZ82jhsdRaXis9xZown46gDsfLF6cGNrAezY91C59hJ2M98R3mPJAGoTWewrlxbZzzeggXuGRr8X//DyWpu9GyXzxCgEAPJLyj7CEyrcBNaZijnMACM2dQ560pLrkrYJdFaktpyGokuOx0AkxDwjSeWLzjIEg4jmTyDB9bJsIM3VUEZD/9mgkg7/j+7/5h/6Ce/32we/iQ//EPyv/4Ig7diQAkT68FJS8cFgjxjXgRxitvhh8IEfkqinz8lB99pHHiu7n6hAiCLvD2ty/eS4ur3emcAIxniL6YU0J+Moh7OvWnKZ5zM1Es4KQDSGLEhZsG+AUojkGskgOkjvSRkowqA2XIL30/4g8CM87Lb8X62jCEqfKII+adHP30+vQDtPnx5A3iifrK0g/X8SrFLx+B8tcDZvR0x00C0NHmwFP1NsgaozqAmc8C6rAYAPbx/uTo35+dvC1oekwiH2aFmsbLpdvLfPSx4GKp7JnQECVvdDE+EtZ1lchPQqr4pK4WLgU91NPxFF1oX/yY0TA1atTp/ExflJE2dmHie2D8QP3ic4FKL3Z+nUHx7JqNqymMNEgx0T9yoCEF5C8KIPAC+Q0AYGsZUP7Iv+xgkAjX7KnF44xL8BIzblZSTZaZJJPQzTErl8wpeWcuyao22vkdmQRbcjdyEoi7szK+Pz7BnIzIJ2tJFzcGDsqMf5xDsRExKPP0MZrIs2iKw1bPntXyMtWPntXzAmoeuYqxihBxHsfzEDMmIj6qZJywFqSb45eVQDKALZ9HwHXqXahytbQsdyYVk7P6gjkYWcV0KL+pnnAOZ6AomJ41FiHfQzQwk9SXOVso+RcTpu9R/wGBKop7ccK6kCZwawerunx6pKudpyrA5Mf9/qBz19mNRgq8KWaPOWNZyJ+u6EN4eNKJviOcgEzcLdJXGZwAqkvfL0roQBSNxuDzcvKTcONC9NL7ol4MSi6LxVcXRpkmRSv38uiXdx/fvKGjNv0vPWrsVEalz+p6lEQV+UWiacF6PGOb6ejlb5/XqDIvCZ+V+rcciJ3qRug2rFcUCAq2BgiTrIyf3gTRzU2Xh/usIUkyCO1NF/T5Cemxx6JQpqsKF2rT0oO7jpZ9Bc/GGC3fDiwnpl16wOEwFR7o2dAzUNIdlUdYL9QLg2h11bvBQawrndUsgrCIaQASqgnXwOxmlWhIUKliD9ajK/rxk4ODimd/OlM0ELmfI/HWZD5fhCO3pSmqJsIdStHg1vxFG/Mm6UQaXfEyH3Q9d1J7zqM6BrTlPyozTMvcR1oOI2Qq1viWSMq2TEbaME4/vDz++GFjWiMUWxxyItHZrcO+dcvpuQ223nuk6sDZrJWL7h5fW2j8ylQcVefwUPPv8p6jkWtjziCVV66zXSaerRMutaTRULssEy19T5nPMdBga1irDl3LBKRik27pVGX/IUQqUwATWrHYVtIu6KzNkFUbGtfikTXwRzbBgbK3YAeivWyJ0xoCf3VXebxKPAwjokdope+KjdBk/MPI0YoJ3+mYIG2QmY+kzkwLOnrG5sa/98bf/N0GdCmJ49+RuBrdympYjewsaPCrAs9WSaF5WWVohbID9q120C7tjv1Kx3cD8gSGe965H7xJFWDmgqChwbhagWq4VIWO3zvtT5WW3JMidO6XiKfhuLot9446AclqjqkuHOJ+ihHBjZw4Ilp1QTrt8UE3Bsc8UTsb4sk4TmxBqSCNcSXia3FW3GxGmTLEC4rL67EW3rWQoa30UIZYOm5ePZMleliuCG0br2thTbdudXW5KCsTLgWluVjE3shAf/82vOM+AAGTuTOl033YlfERFKHeszk0SP0pa2KPrQE7mjHx3sC4uwkYU5sd1i2mYGkZGRXKZE1AUvtSLr1eVjeOGJV8MLJdeS5Vgu9GK+4pFK2abirWW3JfVq1A5/61CEDlxlecn3kGnQaeP3FTyy7HskVeYnVUhT8AImfWLFcR35W63Cw31ten2Uqpw1T1UM4Df0lHZ1sk2tvoAZulFT9fJYW9eXH2CJD90bi0NmuPVPplfPzT84YdGgpWQ0e5nLKjEzHvSvWhLixoo5KfgUHb2Q0ULmLPVIQybh9/V9qkU5mYV1KunTSKsR9dHrnU0qDrYXjY0Ij+VL6GoMCxSGTV6gD/nexF+eY5X9EOucsVVcIgu/uTpd8nwW4jh9dZVkse3yQF6MHsg/MyXy/vowuaKJMPVD2NjYd8YKXnl3eKDtibJnuiKJD6BEAYxVsd2R3ooBJ+sLi5VcelHZL6xcVpdrsFcFfXDi2VLYCujVGnn8O7kwRvISE0sy8WW6Qq1SOStk3dqRqaFS1JqY0jAzqbIzzKMURx1XBODjm0gPO3hBpG8LLVDUROZUWgcd1wR+Q9UxZxpFjyWvrVSByh53Sle9s0Oig/RkDnWTZRR6OSBwdH8eeRjhhNSMNCNVCp4FGofxxPE6BBzf7Crrqv+P3nbEd91+Br+AHv+P5zf7df//7zYP/Jkwf/3x/k/zuSH80IV/A3s8URfsSNP6GLYTsxfkfjVy70K4uwv/Lxbj5aLdOxqMyJzYP6Fn74wpRpgbQvXP5K+S0fPsH4T/D992yndDJ+eQpwB/7vHgKyV/F/98H//wf6/4utF+aJn/nLCSD+s9eWjal9wyAjYkCOf/6aTk9+/nq5CvMAP6eyckMuYmsO3jxOpwuppjoefv2gOOGIoh6+tacrz7WDzHEvQCpB27lptbhloFAlJ5SxTDKjbATjJjGyzIbHtcakp2SarIzi3BgTsuF2H/ThcQOXL2bQ+PyyLIJZS2ik5H0k/xjFUoBcBPKbyNwL6Wn8+PJZRZpwyPs/kpPBm6pUWTxGWcNdTjyXvKGPH59fDrmyyfc3+AhUrKWbwOOp0jS4ATl8bcR6iCp/YbGkAHlefGSxBI4PH07r0W7o/dbf29iQk6AvCAQb3zO5q5HebbWNZR6qRDGtqcVvZTEjVjQVn9H95eXSLd0I4NPBXMKUvo8bkWkHfTo8/EQPFjy/VCer/St3Pvc5scaIq+Fn5MqH9BWe2dy59IP5Ii+KlI/4i2LLBMuvUr8ooT3Tw/NDN8K0NZUh4rOViyfESHH2I1Cdw/iyiO5ToixWRRiEAg1/RrHSGxPW1wEACjutQPBWw3kdIFo2dWOle0GJzICrjtZA07AuEScsY+BQK+QE3givVS7fBHpJOOnLqBRKEElq369J9caz9BatSBuIH9XH8Xt7hZa0TsuUxXYGyo7/GWiOzelWLBsIGfxPfn+TElB2vhj/9wL3q4UA3sn/D/fq8X8Hg70H/v8H8f+XgSsG9uFzYb5z00C8cScZ8P5TzGmZoWXh7HQw/l5MwxiVc1dm1SKKoWe3n4ZBAiICfvm5PBijCwSYXNSf5p2WFMVVkeEUP1K7vz/o9+/NtGlUCMossch2YYJluicMbUK7BuHU4NC4VRqRAoRKaVrjXdTHCJtvkDQjgqXsQSvZDrzu4fL2+oe7hwYFAiar3Hc4DzH9X2OK7hTtOhmbn025bnZxcsIsey9okGUT0/Ax5s76Z2Gl6tNImKwNVxn+2mHxKU3tvPHZqYpqmRkIbeIGy2oGKqaqdWtLUH4HYtj2iRM1m02+e/3kGCIBZ5VEH0QJ1lp2+jxmJFA4UDM2FmM0iwx1coD3mbgqvzYwbYzBnlV90iSf3Bg6t6nmP1tl6IqA/XLkB5G1ZIgXyI+z8kltYUh0yaYu1dHFGX7G2cdK0aUsUxFoGq3mceIkemm6x9awbXYaOPjwvNZt5dV4XVuD86G44NwmANboFji/VDlRcOXPiy/pKXyiHIO6XXndyN3dglomL3KdiUMPQUbRbQAmKssz5sywbkksrCfilpweExhnbpq6RapoJgzwXJIdq5QEuuL0pPNV9H9Oi/h1RIC7+P/uk4b9b/fg4IH//0H8/2fa+t63u2/FBVBBhoReHEVX/OVfnYljwHY9APv5s1OMg26LPARaerkY5L47D/0dreEy9rAIOiSk69HBv97FwO6rxHTOzFdf2ugCtodekdylLYCJC8iPTXa2Pimnh3GqJjZEbt4S+jgzbnAt1tKTaHR5iPjBg4RDsioZIpMQc61Xi3CVSoAkPvh9ghAvuEM7KfeP91pLaIhz5UWsONd/Pv75+OTYeXv88ugNHkc0alGNG53xFHIJlH1nyrEWOyosgkKaaJMptEymYdWkBSVO3TYU1oIlSMg2CFpsnKRRAEiZaoySe2xulcZ9Wm2W63CzHIpcb5W4BKai5+U0aehStvoa4lczVbo7c+gUjlE3HtRtBzw1zHzem09I2KVGKlLIpIgpXgQbiiyKItDQKjM2fGiD1sXmDG26e2Ak5w2SrO+NZE59XXpMWHyEJccvCuDASZdvJA8penvwG3xZ/i8PG30VAeAO/n/Qf7LX8P/tDx74/x/E/49568Xec+D3bhSDWk42ADx6Fa/mCzyagLFZqAnhCQU81vUNqUmn7569wNiu81efKf/ZtCoq1PR6ouAtpPj45P2ro4+nksKjBhNlYZwvFEz29ia9vj3ozTCzvPiLWEVzOkG3DChHQ4w5FHHUpG036nQow15XHL17KT+OhlevPr599q7L2fecZx9fvj7m5/LyBQzFOf7xx9MjPO012P129+C7Lv7t9/nv7mGf/x48kX+/lc8Pfx+vrqyuXLdnqzz+EJ/7UfAZBRK8fYuc5sc4feGuMjd887baSBa5U1UZ96bhzBDKS7GN1YPcJnW7ifS5SOuJ7gWh+AZunE5iVlqfzAaHTrZKEs4KKbvgMrKt6mDzGPlrZQUaRhYCGKvdKNO2WO0NdHkYFXsMdF/4Sto6sKkFs/a9KM4zgHswouVvWoVALvaBpwarz557voNFnd3988Vn47YesVi1w38S+07gZYrpw3ah+FsIKI+AqLqPMAMJGzO6koFD71EWp9nIwKw5dhAlq9yBdmptSuABKeGML7maeXZGGDse06elMrl15dsWJB/Tt8C9YDkabFhkCmnghqLYmaduI08Yp2Rtsw3QINDPEaFDyFm62fmIm0JrqRMG51QGc97EDn8WivSJbYMIl+6VE/mXTo5IkI3KXTkzqq8otfGdvqZtuyULzahhrwGdxc8pW4aT+JEb5tdFmear7Xvz44xngS6cggqXGzQnTyMmoO2Pz2A17WzhJv7ZYDwsrSlLNz1HyDGp8Ein7BZsawT0I665zPA7glirttncG/x/Rm/x82VkVqIPuw+0LsuCePXnUclAxpW8kpRZE4rQh7+e4Me5nlQTSFaPI246dqHYdJLG3mrq41kfaVySEKB94AYTSeeiV+FjdHodbWI44mE0BmSgFK+W9nXXAcjeu/BvjzJRd+W/Sv77oDw/E/GEaidKoOoT/ABZNWZuoPJIT88mY+t76EZ7gAuL35Db7393WIu126sW28Viu9A8Ff2+8X4P3++p99Wmaj3uU4/tRWutHmDRg829HuL7w0ZTKBJRTu8KlbqybBBwpEWvb9kgLCwTPIMOlQ+qJIoW/IrOgZZbo+3X3ZSrYtckRiKzxNHY2tWrux2Pu/t4/v1BeXr4Pfwefg+/h9/D7+H38Hv4Pfz+P/v9PyHr3X4AyAAA"
tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz").extractall(APP, filter="data")
print("worker unpacked to", APP, sorted(os.listdir(APP)))

In [ ]:
# 2 · Start: gateway + Cloudflare tunnel + heartbeat to the MCP server. Keeps running.
import os, runpy, sys
from google.colab import userdata
for k in ("WORKER_SECRET", "REGISTER_URL", "PREWARM", "VOICEBOX_VRAM_GB"):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception:
        pass  # optional secrets may be missing
assert len(os.environ.get("WORKER_SECRET", "")) >= 24, "Add WORKER_SECRET in Secrets (🔑) and allow notebook access"
if not os.environ.get("REGISTER_URL"): print("⚠ REGISTER_URL not set — the MCP server won't find this worker")
sys.argv = ["launcher.py"]
runpy.run_path("/content/voicebox/app/launcher.py", run_name="__main__")

In [ ]:
# (debug) tail the logs — run this in a *second* tab/scratch cell while the worker runs
!tail -n 40 /content/voicebox/logs/gateway.log; ls /content/voicebox/logs